In [ ]:
#Mount from Drive
from google.colab import drive
drive.mount('/content/drive')

#libraries
import os
!pip install mne --quiet
import mne
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import sem

In [ ]:
root_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/'
print(os.listdir(root_dir))
# data_dir = root_dir + 'Data/SpeechRate2025/3'

In [ ]:
import mne

# File path to the .set file
filepath = '/content/drive/MyDrive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/EEGData/10/10_001.set'

# Load raw EEG data from EEGLAB .set file
raw = mne.io.read_raw_eeglab(filepath, preload=True)

# Extract events from annotations
events, event_id = mne.events_from_annotations(raw)

# Plot raw data (interactive browser)
raw.plot(n_channels=10, scalings='auto', title='Raw EEG - Subject 10', show=True)



In [ ]:
#Using MNE (search for how to load data & event files in EEGlab format): 3_002.set and fdt

# -------------------------------
# 1. Load EEGLAB .set/.fdt file
# -------------------------------
raw = mne.io.read_raw_eeglab('/content/drive/MyDrive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/3/data/3_002.set', preload=True)
raw.plot(title='Continuous EEG')

# -------------------------------
# 2. Plot continuous data in 20 chunks
# -------------------------------
data, times = raw.get_data(return_times=True)
n_chunks = 20
chunk_size = data.shape[1] // n_chunks

fig, axs = plt.subplots(n_chunks, 1, figsize=(15, 2.5*n_chunks), sharex=True)
for i in range(n_chunks):
    start = i * chunk_size
    stop = start + chunk_size
    axs[i].plot(times[start:stop], data[0, start:stop])
    axs[i].set_title(f'Chunk {i+1}')
plt.xlabel("Time (s)")
plt.tight_layout()
plt.show()

# -------------------------------
# 3. Extract events from annotations
# -------------------------------
events, event_id_all = mne.events_from_annotations(raw)
print("Available Event IDs:", event_id_all)

# Choose critical event(s) for epoching — adjust as needed
# For example: stimulus onset or task start
event_id = {'Wd1E': 8, 'Wd2E': 10}

# -------------------------------
# 4. Create epochs time-locked to event(s)
# -------------------------------
tmin, tmax = -0.1, 0.4  # 100 ms before,  ms after
epochs = mne.Epochs(raw, events, event_id=event_id, tmin=tmin, tmax=tmax,
                    baseline=(None, 0), preload=True, event_repeated='drop')

# -------------------------------
# 5. Plot each epoch (first channel)
# -------------------------------
epochs_data = epochs.get_data()  # shape: (n_epochs, n_channels, n_times)
times = epochs.times

for i in range(len(epochs_data)):
    plt.plot(times, epochs_data[i, 0, :])
    plt.title(f'Epoch {i+1} - Channel: {raw.ch_names[0]}')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude (μV)')
    plt.show()

# -------------------------------
# 6. Plot ERP ± Confidence Interval (first channel)
# -------------------------------
erp = np.mean(epochs_data[:, 0, :], axis=0)
ci = sem(epochs_data[:, 0, :], axis=0)

plt.plot(times, erp, label='ERP')
plt.fill_between(times, erp - ci, erp + ci, alpha=0.3, label='±CI')
plt.title('ERP ± 95% CI (First Channel)')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude (μV)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# -------------------------------
# Load EEGLAB data
# -------------------------------
raw = mne.io.read_raw_eeglab('/content/drive/MyDrive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/3/data/3_002.set', preload=True)

# Extract events from annotations
events, event_id = mne.events_from_annotations(raw)
print("Available event labels and codes:", event_id)

# -------------------------------
# Define events of interest
# -------------------------------
# Pick relevant ones for imagined speech or stimulus
event_id_of_interest = {'Wd1E': 8, 'Wd2E': 10}

# -------------------------------
# Create epochs
# -------------------------------
epochs = mne.Epochs(raw, events, event_id=event_id_of_interest,
                    tmin=-0.2, tmax=0.8, baseline=(None, 0),
                    preload=True, event_repeated='drop')

# -------------------------------
# Plot readable chunks of raw EEG
# -------------------------------
raw.plot(duration=10, n_channels=10, start=500)

# -------------------------------
# Plot epochs (interactive, clear)
# -------------------------------
epochs.plot(n_epochs=10, n_channels=10)

# -------------------------------
# Plot ERP ± confidence interval (Channel 0)
# -------------------------------
data = epochs.get_data()  # shape: (n_epochs, n_channels, n_times)
times = epochs.times
erp = data[:, 0, :].mean(axis=0)
ci = sem(data[:, 0, :], axis=0)

plt.figure(figsize=(10, 5))
plt.plot(times, erp, label='ERP')
plt.fill_between(times, erp - ci, erp + ci, alpha=0.3, label='±CI')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude (μV)')
plt.title('ERP ± 95% CI (First Channel)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import sem

# -------------------------------------
# 1. Load EEG (.set and .fdt)
# -------------------------------------
eeg_path = '/content/drive/MyDrive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/3/data/3_002.set'
raw = mne.io.read_raw_eeglab(eeg_path, preload=True)
sfreq = raw.info['sfreq']  # e.g., 250 Hz

# -------------------------------------
# 2. Define onset times (in seconds)
# -------------------------------------
# Manually defined for Subject 3_002
onsets_crtn = [22.043, 39.471, 106.749]
onsets_wd2n = [22.161, 39.575, 106.917]
onsets_crte = [445.737, 526.000, 541.607]
onsets_wd2e = [445.837, 526.176, 541.788]

# -------------------------------------
# 3. Convert to sample indices - onset of the critical
# -------------------------------------
samples_crtn = [int(t * sfreq) for t in onsets_crtn]
samples_wd2n = [int(t * sfreq) for t in onsets_wd2n]
samples_crte = [int(t * sfreq) for t in onsets_crte]
samples_wd2e = [int(t * sfreq) for t in onsets_wd2e]

# -------------------------------------
# 4. Build MNE events array
# -------------------------------------
event_id = {'CRTN': 1, 'Wd2N': 2, 'CRTE': 3, 'Wd2E': 4}
events = []

for s in samples_crtn:
    events.append([s, 0, event_id['CRTN']])
for s in samples_wd2n:
    events.append([s, 0, event_id['Wd2N']])
for s in samples_crte:
    events.append([s, 0, event_id['CRTE']])
for s in samples_wd2e:
    events.append([s, 0, event_id['Wd2E']])
events = np.array(events)

# -------------------------------------
# 5. Create epochs (−100 ms to +900 ms)
# -------------------------------------
epochs = mne.Epochs(raw, events, event_id=event_id,
                    tmin=-0.1, tmax=0.9,
                    baseline=(None, 0),
                    preload=True)

# -------------------------------------
# 6. Visualize epochs (interactive plot)
# -------------------------------------
epochs.plot(n_epochs=10, n_channels=10)

# -------------------------------------
# 7. Plot ERP ± CI for each condition (1st channel)
# -------------------------------------
times = epochs.times

for condition in event_id:
    ep_data = epochs[condition].get_data()  # (n_epochs, n_channels, n_times)
    erp = ep_data[:, 0, :].mean(axis=0)     # Use first channel
    ci = sem(ep_data[:, 0, :], axis=0)

    plt.figure(figsize=(10, 4))
    plt.plot(times, erp, label=f'{condition}')
    plt.fill_between(times, erp - ci, erp + ci, alpha=0.3)
    plt.title(f'ERP ± CI: {condition} (Channel: {raw.ch_names[0]})')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude (µV)')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import sem

# -----------------------------------------------
# 1.Load raw EEG data (EEGLAB .set file)
# -----------------------------------------------
eeg_path = '/content/drive/MyDrive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/3/data/3_002.set'
raw = mne.io.read_raw_eeglab(eeg_path, preload=True)
sfreq = raw.info['sfreq']  # Sampling frequency (e.g., 250 Hz)

print(f"Data loaded: {raw.info['nchan']} channels, {raw.n_times / sfreq:.1f} seconds")

# -----------------------------------------------
# 2.Define manually-identified onsets for critical events (in seconds)
# These are typically determined by viewing the audio/EEG together
# and estimating when the function word starts.
# -----------------------------------------------
onsets_crtn = [22.043, 39.471, 106.749]
onsets_wd2n = [22.161, 39.575, 106.917]
onsets_crte = [445.737, 526.000, 541.607]
onsets_wd2e = [445.837, 526.176, 541.788]

# Convert to sample indices
def to_samples(onsets, sfreq):
    return [int(t * sfreq) for t in onsets]

samples_crtn = to_samples(onsets_crtn, sfreq)
samples_wd2n = to_samples(onsets_wd2n, sfreq)
samples_crte = to_samples(onsets_crte, sfreq)
samples_wd2e = to_samples(onsets_wd2e, sfreq)

# -----------------------------------------------
# 3. Create MNE events array
# MNE expects events as [sample, 0, event_id]
# -----------------------------------------------
event_id = {'CRTN': 1, 'Wd2N': 2, 'CRTE': 3, 'Wd2E': 4}
events = []

for s in samples_crtn:
    events.append([s, 0, event_id['CRTN']])
for s in samples_wd2n:
    events.append([s, 0, event_id['Wd2N']])
for s in samples_crte:
    events.append([s, 0, event_id['CRTE']])
for s in samples_wd2e:
    events.append([s, 0, event_id['Wd2E']])

events = np.array(events)
print(f"Total events: {events.shape[0]}")

# -----------------------------------------------
# 4️.Create epochs: time-locked to critical region onsets
# Epoch window: -100 ms to +900 ms around each event
# Baseline correction: using pre-stimulus period (-100 to 0 ms)
# -----------------------------------------------
epochs = mne.Epochs(raw, events, event_id=event_id,
                     tmin=-0.1, tmax=0.9,
                     baseline=(None, 0),
                     preload=True)
print(epochs)

# -----------------------------------------------
# 5. Visualize the raw EEG in chunks (for artifact checking)
# This plots 10 seconds of 10 channels at a time.
# -----------------------------------------------
raw.plot(duration=10, n_channels=10, start=500)

# -----------------------------------------------
# 6.Visualize individual epochs (interactive plot)
# Helpful to see trial-to-trial variability and artifact contamination.
# -----------------------------------------------
epochs.plot(n_epochs=10, n_channels=10)

# -----------------------------------------------
# 7.Plot averaged ERP ± 95% confidence interval (for first channel)
# This mirrors the paper’s approach: visualize early auditory responses (N1/N280)
# and how they differ by condition (fast vs. slowed speech rate).
# -----------------------------------------------
times = epochs.times
channel_idx = 0  # Plot first channel as an example

for cond, eid in event_id.items():
    ep_data = epochs[cond].get_data()  # (n_epochs, n_channels, n_times)
    erp = ep_data[:, channel_idx, :].mean(axis=0)
    ci = sem(ep_data[:, channel_idx, :], axis=0)

    plt.figure(figsize=(10, 4))
    plt.plot(times, erp, label=f'{cond}', color='blue')
    plt.fill_between(times, erp - ci, erp + ci, alpha=0.3, color='lightblue')
    plt.axvline(0, color='gray', linestyle='--', label='Onset (0s)')
    plt.title(f'ERP ± 95% CI: {cond} (Channel: {raw.ch_names[channel_idx]})')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude (µV)')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# ------------------------------------------------
# Summary:
# Load raw EEG data
# Mark critical word onsets and create events
# Create epochs for these events
# Visualize raw data, individual epochs, and averaged ERPs
# Plot confidence intervals to assess variability
# ------------------------------------------------
